In [1]:
"""
Step 4: Training loop for EfficientNet-B2 on cattle/buffalo breed classification (v2 dataset).
Jupyter-notebook friendly (NUM_WORKERS=0, no __main__ guard needed).
 
Paste these cells into your notebook (can split into multiple cells if you like).
"""
 
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
import time

In [2]:
# ==================== CONFIG ====================
CODE_DIR = Path(r"D:\B.Tech\CE-AI\Sem 7\MP\Code work")
IMG_SIZE = 260
BATCH_SIZE = 16
NUM_WORKERS = 0          # Jupyter-safe on Windows
NUM_EPOCHS = 40          # upper cap; early stopping will likely stop sooner
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4      # AdamW-specific regularization
EARLY_STOP_PATIENCE = 6  # stop if val_acc doesn't improve for 6 epochs
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
 
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [3]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
 
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [4]:
class BreedDataset(Dataset):
    def __init__(self, csv_path, class_to_idx, transform=None):
        self.df = pd.read_csv(csv_path)
        self.class_to_idx = class_to_idx
        self.transform = transform
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        label = self.class_to_idx[row["label"]]
        if self.transform:
            image = self.transform(image)
        return image, label

In [5]:
# ==================== LOAD CLASSES & DATA ====================
with open(CODE_DIR / "classes_v2.txt", "r") as f:
    classes = [line.strip() for line in f.readlines()]
NUM_CLASSES = len(classes)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
print(f"Number of classes: {NUM_CLASSES}")
 
train_dataset = BreedDataset(CODE_DIR / "train_v2.csv", class_to_idx, transform=train_transform)
val_dataset = BreedDataset(CODE_DIR / "val_v2.csv", class_to_idx, transform=eval_transform)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
 
print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

Number of classes: 62
Train samples: 9752 | Val samples: 2090


In [6]:
# ==================== CLASS WEIGHTS ====================
train_df = pd.read_csv(CODE_DIR / "train_v2.csv")
train_labels_idx = train_df["label"].map(class_to_idx).values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels_idx
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f"Class weights: min={class_weights.min().item():.2f}, max={class_weights.max().item():.2f}")

Class weights: min=0.36, max=2.15


In [7]:
# ==================== MODEL ====================
model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
model = model.to(DEVICE)

In [8]:
# ==================== LOSS, OPTIMIZER (AdamW), SCALER, SCHEDULER ====================
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
 
print("Setup complete. Ready to train.")

Setup complete. Ready to train.


In [9]:
"""
Step 4b: Training loop with early stopping.
Run this AFTER v2_step4a_setup.py cell (needs model, optimizer, loaders, etc. in memory).
Paste as the next cell in your notebook.
"""
 
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
 
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad()
 
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)
 
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
 
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
 
    return running_loss / total, correct / total

In [10]:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
 
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)
 
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
 
    return running_loss / total, correct / total


In [11]:
best_val_acc = 0.0
epochs_without_improvement = 0
best_model_path = CODE_DIR / "best_model_v2_effnetb2.pth"
history = []
 
print(f"Starting training (max {NUM_EPOCHS} epochs, early stop patience={EARLY_STOP_PATIENCE})...\n")


Starting training (max 40 epochs, early stop patience=6)...



In [ ]:
for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
 
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step(val_acc)
 
    elapsed = time.time() - start_time
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch [{epoch}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
          f"LR: {current_lr:.2e} | Time: {elapsed:.1f}s")
 
    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
        "val_loss": val_loss, "val_acc": val_acc, "lr": current_lr
    })
 
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": classes,
            "val_acc": val_acc,
            "epoch": epoch
        }, best_model_path)
        print(f"  -> New best model saved (val_acc={val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  -> No improvement for {epochs_without_improvement} epoch(s)")
 
    if epochs_without_improvement >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch} "
              f"(no improvement for {EARLY_STOP_PATIENCE} epochs).")
        break

C:\Users\Mahipal\.conda\envs\tf_gpu\Lib\site-packages\PIL\Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [1/40] Train Loss: 2.8120 Acc: 0.2912 | Val Loss: 1.6856 Acc: 0.5196 | LR: 1.00e-04 | Time: 317.4s
  -> New best model saved (val_acc=0.5196)
Epoch [2/40] Train Loss: 1.6114 Acc: 0.5399 | Val Loss: 1.1658 Acc: 0.6397 | LR: 1.00e-04 | Time: 312.4s
  -> New best model saved (val_acc=0.6397)
Epoch [3/40] Train Loss: 1.1766 Acc: 0.6516 | Val Loss: 0.9921 Acc: 0.6828 | LR: 1.00e-04 | Time: 386.7s
  -> New best model saved (val_acc=0.6828)
Epoch [4/40] Train Loss: 0.8943 Acc: 0.7304 | Val Loss: 0.8785 Acc: 0.7177 | LR: 1.00e-04 | Time: 297.6s
  -> New best model saved (val_acc=0.7177)
Epoch [5/40] Train Loss: 0.7133 Acc: 0.7778 | Val Loss: 0.8373 Acc: 0.7407 | LR: 1.00e-04 | Time: 293.4s
  -> New best model saved (val_acc=0.7407)
Epoch [6/40] Train Loss: 0.5718 Acc: 0.8205 | Val Loss: 0.8352 Acc: 0.7641 | LR: 1.00e-04 | Time: 287.4s
  -> New best model saved (val_acc=0.7641)
Epoch [7/40] Train Loss: 0.4917 Acc: 0.8404 | Val Loss: 0.7897 Acc: 0.7703 | LR: 1.00e-04 | Time: 305.0s
  -> Ne